# Forecasting 4 — CRISP-DM Phases 1–6: Business-to-Deployment Workshop

**Business context:** Garden-centre network

**Problem:** Forecast next-week store sales to plan labour and seasonal stock.

**Proposed success criterion:** Beat the previous-week naive forecast MAE by at least 10% on the last 12 weeks; examine seasonal peaks.

**Dataset:** `forecasting_4.csv`. These are synthetic teaching records, not real company data.

**Submission deadline: Monday 5 October 2026 at 13:00 (1:00 PM).** Submit your completed team notebook on Brightspace. Phases 1–2 must be reviewed and updated; your main practical work is Phases 3–6. Any unfinished in-class work must be completed before the deadline.


### Phase 1 – Business understanding
- **Decision:** how many staff hours to roster and how much seasonal stock to order per store for next week.
- **Stakeholder:** store managers (rosters) and the central planning/purchasing team (stock).
- **Prediction moment:** end of week *t−1* (Sunday evening), forecasting total sales for week *t*.
- **Observation unit / key:** one store-week, key = (`store_id`, `week_start`). **Target:** `sales_eur`.
- **Revised success criterion:** the supplied criterion (beat previous-week naive MAE by ≥10% on the last 12 weeks) is kept, but it hides peaks, which is where labour/stock errors cost most. We add: (a) the model must also beat naive on holiday/promo weeks, and (b) no store may be worse than naive, with bias checked (under-forecasting → empty shelves/understaffing).

### Phase 2 – Data understanding
**Findings (evidence below):**
1. Strong seasonality: mean sales peak in Dec–Feb (~€11.9–12.4k) and dip in Jul–Aug (~€9.1k); holiday weeks average ~€12.8k vs ~€10.4k. Calendar features are worth adding.
2. `footfall_actual` (r = 0.72) and `returns_eur_after_week` (r = 0.52) correlate strongly with sales, but both are only known *after* the week → leakage, must be excluded.

**Limitations:**
1. Data-quality issues: one duplicated store-week (S02, 2024-05-20), a negative sales value (S03, −€220), and inconsistent labels (`PROMO`/`promo`, `South`/`south`).
2. Only 6 stores and ~2 years (105 weeks), synthetic data, and no cost information (€ per staff hour, cost of a stock-out) to translate MAE into business value.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
DATA = Path('../datasets/forecasting_4.csv')
if not DATA.exists(): DATA = Path('datasets/forecasting_4.csv')
df = pd.read_csv(DATA)
print(df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(12))
print('Exact duplicates:',df.duplicated().sum())

In [ ]:
# Extra Phase 2 evidence used in the review above
print('Duplicate (store_id, week_start) keys:', df.duplicated(['store_id','week_start']).sum())
print('Category spellings:', df['promo_label'].unique(), df['store_zone'].unique())
print('Sales <= 0:\n', df.loc[df['sales_eur'] <= 0, ['store_id','week_start','sales_eur']])
print(df[['footfall_actual','returns_eur_after_week','holiday_week','sales_eur']].corr()['sales_eur'].round(2))
print(df.groupby(pd.to_datetime(df['week_start']).dt.month)['sales_eur'].mean().round(0))


## Phase 3 — Data Preparation: three justified decisions
For EACH insight: evidence → technique → before/after verification → interpretation → next decision. Consider identifiers, leakage, duplicate keys, suspicious values, categories, date parsing and feature engineering. Explain why you retain, delete or combine variables. **Do not use the target or post-outcome fields as predictors.**

In [ ]:
work = df.copy()

# Decision 1 – SALES (target): one clean, time-ordered sales series per store
work['week_start'] = pd.to_datetime(work['week_start'])
print('Rows before:', len(work), '| duplicate store-weeks:', work.duplicated(['store_id','week_start']).sum())
work = work.drop_duplicates().sort_values(['store_id','week_start']).reset_index(drop=True)
print('Rows after: ', len(work), '| duplicate store-weeks:', work.duplicated(['store_id','week_start']).sum())
bad = work['sales_eur'] < 0                                   # -€220 is impossible weekly sales
print('Negative sales weeks:', bad.sum())
work.loc[bad, 'sales_eur'] = np.nan
work['sales_eur'] = work.groupby('store_id')['sales_eur'].transform(lambda s: s.interpolate())
print('Negative/missing sales after:', (work['sales_eur'] < 0).sum(), work['sales_eur'].isna().sum())

# Decision 2 – WEATHER: rain_mm_actual is only known AFTER the week, so it cannot be used as is.
#   (a) rain_last_week  = last week's observed rain (known at the prediction moment)
#   (b) rain_forecast   = stand-in for a next-week weather forecast (we use the actual rain as a
#       "perfect forecast" proxy → optimistic upper bound for what weather can add)
#   (c) heavy_rain_fc   = forecast of ≥15 mm in the week (garden-centre visits drop in very wet weeks)
print('Rain (mm) per week:', work['rain_mm_actual'].describe()[['min','mean','max']].round(1).to_dict())
gs = work.groupby('store_id')
work['rain_last_week'] = gs['rain_mm_actual'].shift(1)
work['rain_forecast']  = work['rain_mm_actual']
work['heavy_rain_fc']  = (work['rain_forecast'] >= 15).astype(int)
display(work.groupby(pd.cut(work['rain_mm_actual'], [-1,2,6,10,15,30]), observed=True)['sales_eur']
            .agg(['mean','count']).round(0).rename_axis('rain_mm'))

# Decision 3 – PREDICTION features: past sales + calendar, built with shift() so the
# forecast for week t only uses information up to week t-1
g = work.groupby('store_id')['sales_eur']
work['sales_lag1']  = g.shift(1)                                     # last week's sales
work['sales_lag2']  = g.shift(2)
work['sales_roll4'] = g.transform(lambda s: s.shift(1).rolling(4).mean())   # recent sales level
woy = work['week_start'].dt.isocalendar().week.astype(int)
work['season_sin'] = np.sin(2*np.pi*woy/52); work['season_cos'] = np.cos(2*np.pi*woy/52)
work['promo'] = work['promo_label'].str.lower().eq('promo').astype(int)    # fixes PROMO/promo
# Excluded: footfall_actual, returns_eur_after_week, average_price_eur (only known after/during
# the week → leakage); store_size_category and store_zone (store_id already captures the store).
display(work[['store_id','week_start','sales_eur','sales_lag1','sales_roll4','rain_last_week','rain_forecast']].head(3))


### Preparation decision log — sales, weather and prediction
| Phase 2 finding | Technique and reason | Verification | Risk | Next decision |
|---|---|---|---|---|
| **Sales:** 1 duplicated store-week (S02, 2024-05-20) and an impossible −€220 sales week (S03, 2025-03-03) | Drop the duplicate, sort per store by week, set the negative value to missing and interpolate within the store – a forecast needs one clean, continuous sales series per store | Rows 631 → 630; duplicate store-weeks 1 → 0; negative sales 1 → 0 | Medium – true value of the −€220 week unknown (refund booking?) | Ask finance; keep the cleaned series as target |
| **Weather:** `rain_mm_actual` is measured *after* the week, so using it directly is leakage. Sales are lower in very wet weeks (≥15 mm: ~€9.5k vs ~€10.6k), but only 27 such weeks | Build weather features that exist on Sunday: `rain_last_week` (lag 1) and `rain_forecast` (actual rain as a stand-in for a weather forecast = optimistic upper bound) plus a `heavy_rain_fc` flag | Rain bins vs sales table above; no current-week actual is used except as forecast proxy | Proxy overstates forecast accuracy – a real 7-day forecast is less precise | Test in Phase 4 whether weather improves the sales forecast |
| **Prediction:** sales depend on the recent level and season (Dec–Feb peak, Jul–Aug dip); `PROMO`/`promo` spelled inconsistently; footfall and returns are post-outcome | Lagged sales (`sales_lag1`, `sales_lag2`, `sales_roll4`) with `shift()` so only data up to week *t−1* is used; season sin/cos, `holiday_week`, harmonised `promo` flag. **Excluded:** footfall, returns, average price (leakage) | First 4 weeks per store have no history and are dropped | Promo assumed planned in advance | Use these as the core forecasting features |

## Phase 4 — Modelling: forecasting next week's sales
- **Prediction moment:** Sunday evening after week *t−1*; we forecast total `sales_eur` for week *t* per store (1-week-ahead forecast).
- **Forecast drivers allowed at that moment:**
  - *Sales history* – last week's sales, the week before, 4-week rolling average.
  - *Weather* – last week's rain and the rain forecast for next week (proxy, see Phase 3).
  - *Calendar* – season (week of year), holiday week, planned promotion, store size and `store_id`.
- **Excluded:** `footfall_actual`, `returns_eur_after_week`, `average_price_eur` (only known after/during the week).
- **Validation:** time-ordered – train on weeks up to 6 Oct 2025, forecast the last 12 weeks (13 Oct – 29 Dec 2025), never shuffling. Scaling and encoding are learned on the training weeks only (`Pipeline`).
- **Forecasts compared:** (1) naive baseline *next week = last week*, (2) Ridge on sales + calendar, (3) Ridge on sales + calendar + **weather**, to measure how much weather really adds.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

SALES   = ['sales_lag1','sales_lag2','sales_roll4']
CAL     = ['season_sin','season_cos','holiday_week','promo','store_floor_m2']
WEATHER = ['rain_last_week','rain_forecast','heavy_rain_fc']
model_df = work.dropna(subset=SALES + WEATHER).copy()          # first 4 weeks per store lack history

# Chronological split: forecast the last 12 weeks, train on everything before
cutoff = model_df['week_start'].sort_values().unique()[-12]
train = model_df[model_df['week_start'] < cutoff]
test  = model_df[model_df['week_start'] >= cutoff].copy()
print('Train:', train['week_start'].min().date(), '→', train['week_start'].max().date(), len(train))
print('Test: ', test['week_start'].min().date(), '→', test['week_start'].max().date(), len(test))

def forecaster(num):
    """Ridge forecaster; scaling/encoding is learned on the training weeks only."""
    pipe = Pipeline([('prep', ColumnTransformer([('num', StandardScaler(), num),
                                                 ('cat', OneHotEncoder(handle_unknown='ignore'), ['store_id'])])),
                     ('ridge', Ridge(alpha=1.0))])
    return pipe.fit(train[num + ['store_id']], train['sales_eur'])

m_sales   = forecaster(SALES + CAL)              # sales history + calendar
m_weather = forecaster(SALES + CAL + WEATHER)    # + weather
test['pred_naive']   = test['sales_lag1']        # baseline: next week = last week
test['pred_sales']   = m_sales.predict(test[SALES + CAL + ['store_id']])
test['pred_weather'] = m_weather.predict(test[SALES + CAL + WEATHER + ['store_id']])

coef = pd.Series(m_weather.named_steps['ridge'].coef_[:len(SALES+CAL+WEATHER)], index=SALES+CAL+WEATHER)
print('Effect per 1 SD change (€/week):'); display(coef.round(0).sort_values())


In [ ]:
mae = lambda d, c: mean_absolute_error(d['sales_eur'], d[c])
preds = {'Naive (last week)':'pred_naive', 'Sales + calendar':'pred_sales', 'Sales + calendar + weather':'pred_weather'}
res = pd.DataFrame({k: {'MAE (€)': mae(test, c)} for k, c in preds.items()}).T
res['vs naive'] = (1 - res['MAE (€)'] / res.loc['Naive (last week)','MAE (€)']).map('{:.1%}'.format)
display(res.round(0))

test['peak']    = np.where(test['holiday_week'].eq(1) | test['promo'].eq(1), 'holiday/promo', 'normal')
test['weather'] = np.where(test['rain_forecast'] >= 10, 'wet (≥10 mm)', 'dry (<10 mm)')
def seg(col):
    return test.groupby(col).apply(lambda d: pd.Series(
        {'weeks': len(d), **{k: mae(d, c) for k, c in preds.items()},
         'bias (weather model)': (d['pred_weather'] - d['sales_eur']).mean()}), include_groups=False).round(0)
display(seg('peak')); display(seg('weather')); display(seg('store_id'))


## Phase 5 — Evaluation of the sales forecasts
| Forecast MAE on last 12 weeks (72 store-weeks) | Overall | Holiday/promo | Dry weeks (<10 mm) | Wet weeks (≥10 mm) |
|---|---|---|---|---|
| Naive (last week) | €1,058 | €1,185 | €1,135 | €884 |
| Sales + calendar | €705 (−33%) | €785 | €673 | €778 |
| Sales + calendar + weather | **€701 (−34%)** | €788 | €665 | €782 |

- **Sales:** sales history and season drive the forecast. The 4-week rolling sales level, store size and season have the largest effects; both Ridge forecasts beat the naive forecast by ~33%, well past the 10% target, including on holiday/promo peak weeks.
- **Weather:** adding rain improves the forecast by only €4 per store-week (<1%), even with a "perfect" rain forecast. The weather effects are small (≈ −€14 to +€40 per SD of rain). Wet weeks are forecast too high (bias +€258) – the dip in very wet weeks is real but too rare (27 weeks in two years) for the model to learn it.
- **Prediction risk by store:** S01 is still worse than naive (MAE €867 vs €771, bias +€548 → over-forecasting means overstaffing/overstock at the smallest store).
- **Is the criterion realistic?** Yes, but too easy: a naive forecast ignores season and holidays. A seasonal-naive benchmark and a longer rolling backtest (incl. the spring and summer weather) would be stricter; our 12 test weeks are Oct–Dec only.
- **Assumption / data request:** no cost data, so over- and under-forecasts are weighted equally. We also need real historical *weather forecasts* (not actuals) to evaluate the weather feature honestly.

## Phase 6 — Deployment proposal (design only)
**Who / action:** store managers use the next-week sales forecast to roster staff; central purchasing sums it across stores for seasonal stock orders.
**When it runs:** every Sunday at 20:00, after the week's sales are closed. The job (1) loads last week's sales, (2) pulls the 7-day **rain forecast** per store location from a weather service, (3) rebuilds the sales-lag, weather and calendar features and (4) publishes a forecast per store with a range (± test MAE ≈ €700).
**Human override:** managers can adjust the forecast for things the model cannot see (local event, extreme weather warning, road works) and must log a reason; overrides are compared with actual sales.
**Monitoring:** weekly 4-week rolling MAE and bias per store against the naive forecast, plus MAE split into wet vs dry weeks to check the weather effect.
**Refresh trigger:** retrain monthly, or at once if the 4-week MAE is >20% above €700 or the weather-forecast feed changes.
**Pause condition:** if a store's forecast is worse than naive for 4 consecutive weeks (S01 is currently there) or the weather feed is missing, that store falls back to the naive/sales-only forecast until reviewed.

## Final decision + individual accountability
**Decision: Pilot** in stores S02–S06, and **Improve** the forecast for S01 before using it there.
Evidence: the sales + calendar + weather forecast lowers the average forecast error (MAE) from €1,058 (naive, "same as last week") to €701, a 34% improvement and well above the 10% target. It also wins in holiday/promo peak weeks, and no leaking fields are used. However, weather adds less than 1% (€705 → €701), wet weeks are over-forecast by about €258, and S01 is still worse than naive (MAE €867 vs €771, bias +€548). S01 therefore stays on the naive forecast while the model is improved, and real historical weather forecasts and cost data are needed before a full rollout.

*Ivan van Vreeswijk*: Preparation choice – Store S03 had a sales value of −€220 in the week of 3 March 2025, which is impossible for a garden centre. I set it to missing and filled it with the average of the weeks around it, so the sales history the forecast uses has no gaps or impossible values. Model finding – Adding rain to the model barely helped: the error only dropped from €705 to €701 per store-week. In wet weeks the forecast was also too high, because very rainy weeks are rare in the data. Business implication – On its own, rain is not a reason to change the forecast. A manager who expects a very wet week should still be allowed to lower it by hand.

*David Malena:* Preparation choice: I dropped footfall_actual, returns_eur_after_week, and average_price_eur because they're only known after the week ends, avoiding data leakage during Sunday planning. Model finding: Past sales and seasonality did pretty much all the heavy lifting in Ridge, especially the 4-week rolling average and store size, and it beat naive across both normal and peak weeks. Business implication: Managers can plan rosters and stock orders using just internal historical sales data, so there's no need to pay for external footfall forecasts.

*Lyndon Boesveld*: Preparation choice – Store S02 had the week of 20 May 2024 recorded twice, and the promo column mixed "PROMO" and "promo". I removed the duplicate (631 → 630 rows) and merged the spellings into one promo flag, so each store-week counts once and every promotion is recognised. Model finding – The model beats naive in five of the six stores, but not in S01, the smallest store: its error is €867 against €771 for naive, and it forecasts about €548 too high on average. Business implication – S01 should stay on the "same as last week" forecast to avoid overstaffing and overstock, so the pilot runs in S02–S06 until S01's forecast beats naive.

*Momtez Hlayem*: Preparation choice: I cleaned the sales data by removing the duplicated store-week, correcting the inconsistent promotion labels, and replacing the impossible negative sales value with a missing value before forecasting. Model finding: The sales + calendar model reduced MAE from €1,058 for the naive forecast to €705, while adding weather only improved it slightly to €701. Business implication: The model can help managers plan staff and stock, but S01 should still use the naive forecast because the model performs worse there.